
# Step-by-Step ChromaDB Tutorial (Production-Oriented)

This notebook provides a **step-by-step introduction to ChromaDB**, a popular open‑source vector database used in Retrieval Augmented Generation (RAG) systems.

The tutorial gradually moves from **basic concepts to advanced usage**, including:

- Creating and managing vector collections
- Storing embeddings and documents
- Performing semantic search
- Using metadata filters
- Custom embedding models
- Updating and deleting documents
- Building a simple RAG retrieval pipeline

Notebook generated on: **2026-03-12**



## Step 1 — Install ChromaDB

Create a fresh Python environment and install required dependencies.

Required packages:

- chromadb
- sentence-transformers
- pandas (optional for data manipulation)
- langchain (optional for integration with LLM pipelines)

Run the following commands in your terminal before running the notebook.


In [ ]:

# Run these in a terminal if not already installed:
# pip install chromadb
# pip install sentence-transformers
# pip install pandas
# pip install langchain



## Step 2 — Initialize ChromaDB

ChromaDB supports two main modes:

1. **In-memory database** (good for experiments)
2. **Persistent database** (recommended for real applications)

Below we initialize a simple in-memory client.


In [1]:

import chromadb

client = chromadb.Client()

print("ChromaDB client initialized")


ChromaDB client initialized



## Step 3 — Create a Collection

A **collection** in ChromaDB is similar to a table in a relational database.

Each collection stores:

- documents
- embeddings
- metadata
- ids

We create a collection for company policy documents.


In [2]:

collection = client.create_collection(name="company_policies")

print("Collection created")


Collection created



## Step 4 — Insert Documents

Let’s simulate a small **enterprise policy dataset**.

These documents could represent:

- HR policies
- security policies
- remote work rules


In [3]:

documents = [
    "Employees are entitled to 20 days of annual leave per year.",
    "Travel reimbursement requires manager approval.",
    "Passwords must be changed every 90 days.",
    "Employees can work remotely two days per week.",
    "Maternity leave is available for 26 weeks."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

collection.add(
    documents=documents,
    ids=ids
)

print("Documents inserted into ChromaDB")


Documents inserted into ChromaDB



## Step 5 — Perform Semantic Search

One of the key advantages of vector databases is **semantic similarity search**.

Instead of matching keywords, the system retrieves documents based on **meaning**.


In [4]:

results = collection.query(
    query_texts=["How many leave days do employees get?"],
    n_results=2
)

results["documents"]


[['Employees are entitled to 20 days of annual leave per year.',
  'Employees can work remotely two days per week.']]


## Step 6 — Use Metadata

Metadata allows us to attach structured information to each document.

Example metadata:

- department
- document type
- year


In [5]:

collection.add(
    documents=[
        "Employees get 20 days of annual leave",
        "Travel expenses require approval",
        "Password must change every 90 days"
    ],
    ids=["m1", "m2", "m3"],
    metadatas=[
        {"department": "HR"},
        {"department": "Finance"},
        {"department": "IT"}
    ]
)

print("Documents with metadata inserted")


Documents with metadata inserted



## Step 7 — Query with Metadata Filtering

Metadata filtering helps restrict search results to **specific categories**.


In [7]:

results = collection.query(
    query_texts=["leave policy"],
    where={"department": "IT"}  # Experiment without this line and justify the results
)

results["documents"]


[['Password must change every 90 days']]


## Step 8 — Use Custom Embedding Models

Instead of default embeddings, we can use models from **Sentence Transformers**.


In [ ]:
!pip3 install sentence-transformers

In [ ]:
import sys
sys.executable

In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded



## Step 9 — Create Collection with Custom Embeddings

We define a function that converts text into embeddings.


In [ ]:

'''
# Older method creates errors
custom_collection = client.create_collection(
    name="policy_docs",
    embedding_function=embed
)
'''
'''
custom_collection = client.get_or_create_collection(
    name="policy_docs",
    embedding_function=embed
)

'''


In [10]:
model.encode("Leave policy")

array([ 5.40542975e-02,  6.17161766e-02,  1.66892540e-02, -1.86278345e-03,
        1.58608332e-01,  9.33696553e-02,  7.48863518e-02, -8.87880921e-02,
       -4.96507585e-02,  1.16950413e-02,  4.25149798e-02,  2.06487160e-02,
       -2.89833769e-02,  1.04336133e-02,  3.46548371e-02,  6.13906085e-02,
       -3.72690596e-02, -4.46319431e-02, -1.11291118e-01, -4.53270786e-02,
       -3.80912870e-02, -1.91865973e-02, -2.91834213e-02,  3.85390669e-02,
       -6.66598650e-03,  6.34843558e-02, -2.37590028e-03,  2.79673487e-02,
       -2.22749785e-02,  1.28877843e-02,  6.74123988e-02, -1.36151714e-02,
       -5.99818081e-02, -3.78925651e-02, -5.95202763e-03,  3.82349603e-02,
       -8.27090293e-02, -7.12215975e-02, -1.46410307e-02, -1.72868185e-02,
       -4.86825797e-04,  3.56111745e-03,  1.35520156e-02,  3.40271704e-02,
        4.50222827e-02,  6.66233823e-02, -2.42052847e-04, -4.38281894e-02,
        2.65351497e-02,  6.86439723e-02,  5.65922894e-02,  1.20148240e-02,
        1.55096631e-02,  

In [11]:
class MyEmbeddingFunction:
    def __init__(self, model):
        self.model = model

    def __call__(self, input): # __add__
        # Used for documents
        return self.model.encode(input).tolist()

    # important for queries
    def embed_query(self, input):
        # Used for queries
        return self.model.encode(input).tolist()


# emb_fun = lambda input : model.encode(input).tolist()

embedding_fn = MyEmbeddingFunction(model)

# client.delete_collection(name="policy_docs")
custom_collection = client.create_collection(
    name="policy_docs",
    embedding_function=embedding_fn
)


## Step 10 — Batch Insert Documents

For large datasets we typically insert documents in batches.


In [12]:

documents_batch = [
    "Annual leave policy for employees.",
    "Expense reimbursement guidelines.",
    "Security and password management rules.",
    "Remote work guidelines."
]

ids_batch = [f"id{i}" for i in range(len(documents_batch))]

custom_collection.add(
    documents=documents_batch,
    ids=ids_batch
)

print("Batch documents inserted")


Batch documents inserted



## Step 11 — Store Embeddings Directly

Sometimes embeddings are generated externally (for example using OpenAI or other APIs).


In [13]:

embeddings = model.encode(documents_batch).tolist()
print("Embeddings: ", embeddings)


Embeddings:  [[0.03157561644911766, 0.06579718738794327, 0.03483077883720398, 0.021718231961131096, 0.07771220803260803, 0.10137168318033218, 0.007153652608394623, -0.10173303633928299, -0.04853944852948189, -0.023728778585791588, 0.051852818578481674, 0.027457967400550842, -0.07345768064260483, -0.0004092139715794474, 0.012610329315066338, 0.009944936260581017, -0.0053214929066598415, 0.008577434346079826, 0.01283390074968338, -0.07869306206703186, -0.007211841177195311, -0.050749629735946655, -0.044629041105508804, 0.034337639808654785, -0.019533729180693626, 0.02587764523923397, 0.003106894204393029, 0.028237201273441315, -0.041118599474430084, 0.07553992420434952, 0.01802549511194229, 0.010595862753689289, -0.017014101147651672, -0.04967638850212097, -0.01767546683549881, 0.021661171689629555, -0.0452861525118351, -0.01774381473660469, -0.02566659450531006, 0.02808370627462864, -0.04783061146736145, 0.01101827621459961, 0.038157131522893906, 0.008658109232783318, 0.0008044176502153

In [14]:
custom_collection.add(
    embeddings=embeddings,
    documents=documents_batch,
    ids=[f"emb{i}" for i in range(len(documents_batch))]
)

print("Documents with external embeddings stored")

Documents with external embeddings stored



## Step 12 — Update and Delete Documents

ChromaDB supports updating or deleting documents.


In [15]:

# Update
custom_collection.update(
    ids=["id0"],
    documents=["Employees get 25 days of annual leave per year"]
)


# Delete

custom_collection.delete(ids=["id1"])

print("Update and delete operations completed")


Update and delete operations completed



## Step 13 — Inspect Collection

Useful commands to inspect collections.


In [16]:

print("Document count:", custom_collection.count())

print("Peek documents:")
custom_collection.peek()


Document count: 7
Peek documents:


{'ids': ['id0', 'id2', 'id3', 'emb0', 'emb1', 'emb2', 'emb3'],
 'embeddings': array([[ 0.07038204,  0.01818218,  0.05550463, ..., -0.02345404,
          0.04196787,  0.01994185],
        [-0.02668817,  0.01933627, -0.08029221, ...,  0.0419888 ,
          0.02214987, -0.026489  ],
        [-0.02874338,  0.00097479,  0.02530235, ...,  0.06525035,
         -0.11450793,  0.02154846],
        ...,
        [-0.03313634,  0.07145615, -0.00087072, ..., -0.02557479,
          0.01851058, -0.07042725],
        [-0.02668817,  0.01933627, -0.08029221, ...,  0.0419888 ,
          0.02214987, -0.026489  ],
        [-0.02874338,  0.00097479,  0.02530235, ...,  0.06525035,
         -0.11450793,  0.02154846]], shape=(7, 384)),
 'documents': ['Employees get 25 days of annual leave per year',
  'Security and password management rules.',
  'Remote work guidelines.',
  'Annual leave policy for employees.',
  'Expense reimbursement guidelines.',
  'Security and password management rules.',
  'Remote work gu


## Step 14 — Build a Simple Retrieval Pipeline

In a RAG system, retrieval works as:

1. User query
2. Embedding generation
3. Vector search
4. Return top documents


In [17]:

query = "What is the leave policy?"

results = custom_collection.query(
    query_texts=[query],
    n_results=3
)

context = results["documents"]

context


[['Annual leave policy for employees.',
  'Employees get 25 days of annual leave per year',
  'Remote work guidelines.']]


## Step 15 — Persistent Storage

For production systems, the vector database must persist on disk.


In [18]:

import chromadb

persistent_client = chromadb.PersistentClient(path="./chroma_db")

collection = persistent_client.get_or_create_collection("persistent_docs")

collection.add(
    documents=["Example persistent document"],
    ids=["p1"]
)

print("✅ Database persisted automatically")


✅ Database persisted automatically
